# 画像前処理パイプライン可視化ツール

このノートブックは、学習に使用する `transforms.Compose` の各ステップ（クロップ、リサイズ、色調変化など）を、データセットの画像を使って可視化し、確認するためのツールです。表示形式は **3列 x 2段のグリッド** を使用します。

### 1. 準備とライブラリのインポート

In [14]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# 補助スクリプトを読み込み (ノートブックと同じディレクトリにある前提)
try:
    from xy_dataset import XYDataset
except ImportError:
    print("エラー: xy_dataset.py が見つかりません。パスを確認してください。")
    # 実行を停止せずに、後でエラーメッセージを出す

# --- グローバル設定 (15_3_train_on_desktop_画像前処理追加.ipynb と共通) ---
DATASET_DIR = 'datasets'
IMAGE_INDEX = 0  # データセットの0番目の画像をテストに使用

print("✅ 準備が完了しました。")

✅ 準備が完了しました。


### 2. データセット選択UI

In [ ]:
# --- データセットスキャン関数 ---
def find_xy_path(root_dir):
    """'xy'フォルダを含む親ディレクトリのパスを返す"""
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if 'xy' in dirnames:
            return dirpath
    return None

def scan_datasets():
    """DATASET_DIRをスキャンして、有効なデータセットパスをリストアップする"""
    found_datasets = []
    if not os.path.exists(DATASET_DIR):
        print(f"警告: ディレクトリ '{DATASET_DIR}' が見つかりません。")
        return []
        
    for item in os.listdir(DATASET_DIR):
        item_path = os.path.join(DATASET_DIR, item)
        if os.path.isdir(item_path):
            final_path = find_xy_path(item_path)
            if final_path:
                found_datasets.append(final_path)
    
    return sorted(found_datasets)

# --- UIウィジェット ---
dataset_options = scan_datasets()
dataset_dropdown = widgets.Dropdown(
    options=dataset_options,
    description='データセット選択:',
    disabled=not bool(dataset_options),
    layout=widgets.Layout(width='80%')
)
visualize_new_button = widgets.Button(
    description='【現行】クロップ処理を可視化',
    button_style='success',
    icon='cut'
)
visualize_old_button = widgets.Button(
    description='【旧型】従来処理を可視化',
    button_style='info',
    icon='history'
)
output_area = widgets.Output()

display(widgets.VBox([
    dataset_dropdown,
    widgets.HBox([visualize_new_button, visualize_old_button]),
    output_area
]))


### 3. 可視化ロジック（共通関数）

各パイプラインのステップを定義し、表示する共通ロジックです。

In [20]:
# --- 処理定義したTransformsのリスト (再利用できるように定義) ---
resize = transforms.Resize((224, 224))
color_jitter = transforms.ColorJitter(0.2, 0.2, 0.2, 0.2)
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
crop_lambda = transforms.Lambda(lambda x: x.crop((0, 75, 224, 224)))

def apply_and_visualize(dataset_path, initial_img, pipeline_type):
    """指定されたパイプラインを適用し、Matplotlibで表示する"""
    
    # 現行のクロップ処理ありパイプライン
    if pipeline_type == 'new':
        steps = [
            ("1. Original Image (224x224)", None, 'PIL'),
            ("2. After Crop (149x224)", crop_lambda, 'PIL'),
            ("3. After Resize (224x224)", resize, 'PIL'),
            ("4. After ColorJitter", color_jitter, 'PIL'),
            ("5. After ToTensor (0~1)", to_tensor, 'Tensor'),
            ("6. After Normalize (Final Input)", normalize, 'Tensor')
        ]
        title_prefix = "Preprocessing:ROI Cropping and Resizing"
        
    # 旧型のクロップ処理なしパイプライン
    elif pipeline_type == 'old':
        steps = [
            ("1. Original Image (224x224)", None, 'PIL'),
            ("2. After Resize (224x224)", resize, 'PIL'), # 旧型はResizeが最初
            ("3. After ColorJitter", color_jitter, 'PIL'),
            ("4. After ToTensor (0~1)", to_tensor, 'Tensor'),
            ("5. After Normalize (Final Input)", normalize, 'Tensor'),
            ("6. (Unused)", None, 'PIL') # 描画数を合わせるためのダミー
        ]
        title_prefix = "Preprocessing:Full-Frame Resizing"
        
    else:
        return
        
    # Matplotlibのセットアップ
    fig, axes = plt.subplots(2, 4, figsize=(15, 10))
    axes = axes.flatten() # 1次元配列に平坦化して扱いやすくする
    
    # fig.suptitle(f"{title_prefix} - {os.path.basename(dataset_path)} (Index: {IMAGE_INDEX})", fontsize=16)
    fig.suptitle(f"{title_prefix}", fontsize=16)
    
    current_img = initial_img

    for i, (title, transform, output_type) in enumerate(steps):
        ax = axes[i]
        display_img = None
        
        if transform is None and i == 0: # 0. Original Image
            display_img = initial_img
        
        elif transform is None and title == "5. (Unused)": # 旧型パイプラインのダミーセル
            # ax.text(0.5, 0.5, '【このステップは未使用】', horizontalalignment='center', verticalalignment='center', fontsize=12, color='gray')
            ax.set_title(title, fontsize=10)
            ax.axis('off')
            continue
        
        # PIL処理: Crop, Resize, ColorJitter
        elif output_type == 'PIL':
            current_img = transform(current_img)
            display_img = current_img
        
        # Tensor処理: ToTensor, Normalize
        elif output_type == 'Tensor':
            if 'ToTensor' in title:
                current_img = transform(current_img) # PIL -> Tensor
                # 表示用にTensorを (H, W, C) に変換 (値は0〜1)
                display_img = current_img.permute(1, 2, 0).numpy()
            elif 'Normalize' in title:
                # ToTensor後のTensor (current_img) に対してNormalizeを適用
                display_img = transform(current_img).permute(1, 2, 0).numpy()

        # --- 画像表示と装飾 ---
        if display_img is not None:
            if isinstance(display_img, Image.Image):
                ax.imshow(display_img, aspect='auto')
            else:
                ax.imshow(display_img)
            
            # 軸メモリとタイトルを設定
            ax.set_title(title, fontsize=10)
            # ax.set_xlabel('Width (0-224)')
            # ax.set_ylabel('Height (0-224)')
            ax.tick_params(axis='both', which='both', length=3, width=1, colors='black')
            # ax.grid(True, linestyle='--', alpha=0.5)
            ax.set_aspect('equal', adjustable='box')
            
            # Crop後の画像はY軸の範囲が異なるため調整
            if pipeline_type == 'new' and i == 1: # 現行パイプラインのCrop後
                # ax.set_ylim(149, 0) # 逆順にして上から表示
                ax.set_yticks([0, 75, 149])
                # ax.set_ylabel('Height (0-149)')
            elif i <= 3: # その他のPIL/Tensor画像 (Original, Resize, Jitter)
                # ax.set_ylim(224, 0) # 逆順にして上から表示
                pass

        
    plt.tight_layout()
    plt.show()


def execute_visualization(pipeline_type):
    """UIイベントから呼び出されるラッパー関数"""
    with output_area:
        clear_output(wait=True)
        
        dataset_path = dataset_dropdown.value
        if not dataset_path:
            print("エラー: データセットが選択されていません。")
            return

        try:
            # 1. 最初の画像（PIL Image）を取得
            image_files = sorted(glob.glob(os.path.join(dataset_path, 'xy', '*.jpg')))
            if not image_files:
                print(f"エラー: '{dataset_path}/xy' に画像ファイルが見つかりません。")
                return
                
            initial_img = Image.open(image_files[IMAGE_INDEX]).convert('RGB')
            
            # 2. 可視化実行
            apply_and_visualize(dataset_path, initial_img, pipeline_type)

        except Exception as e:
            print(f"画像の処理中にエラーが発生しました: {e}")
            import traceback
            print(traceback.format_exc())

# --- イベントリスナーを接続 ---
visualize_new_button.on_click(lambda b: execute_visualization('new'))
visualize_old_button.on_click(lambda b: execute_visualization('old'))

### 4. 実行セクション (UIはセクション2にあります)

In [ ]:
display(widgets.VBox([
    widgets.HBox([visualize_new_button, visualize_old_button]),
    output_area
]))